# Day 3 v2 — Section 5: MAE Optimization + Hypothesis Test

**Mục tiêu:** Cải thiện MAE từ baseline 108.8k (LGB char_wb 50K) xuống thấp nhất có thể.

**Sections:**
- 5A. LGB + char_wb TF-IDF — full 269K train (MSE objective)
- 5B. LGB + char_wb TF-IDF — full 269K train (MAE objective = regression_l1)
- 5C. Blend 5A + 5B — optimize weight trên val set
- 5D. RandomForest + BoW 2000 — hypothesis test (so sánh với English day3)
- 5E. XGBoost + BoW 2000 — hypothesis test (so sánh với English day3)

**Baseline (Section 0-4):** Best = 3B. LGB+char_wb 50K subset → MAE=108.8k, R²=59.6%

**Config máy:** i5-14600KF 18 cores / i7-12700K 20 cores | RAM 28GB | CUDA

In [1]:
import sys, json
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestRegressor

sys.path.append("..")
from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"train={len(train):,} | val={len(val):,} | test={len(test):,}")

documents   = [item.summary for item in train]
prices      = np.array([item.price for item in train], dtype=np.float32)
docs_val    = [item.summary for item in val]
prices_val  = np.array([item.price for item in val], dtype=np.float32)

train=269,112 | val=3,926 | test=3,872


In [3]:
# Ket qua Section 0-4 (hard-coded tu session truoc)
results = {
    "2b. LR + BoW":              {"mae": 131.7, "mse": 31955, "r2": 0.443},
    "2c. Ridge + TF-IDF char_wb":{"mae": 112.1, "mse": 23401, "r2": 0.592},
    "3A. Bench BoW + LGB":       {"mae": 120.8, "mse": 28226, "r2": 0.508},
    "3B. Bench char_wb + LGB":   {"mae": 108.8, "mse": 23173, "r2": 0.596},
    "3C. Bench Underthesea+LGB": {"mae": 118.0, "mse": 28239, "r2": 0.508},
    "4a. RandomForest (15K)": {"mae": 129.7, "mse": None, "r2": 0.423},
    "4b. XGBoost (269K)": {"mae": 125.2, "mse": None, "r2": 0.435},
}
print("Baseline results loaded:", len(results), "models")

Baseline results loaded: 7 models


## Section 5A — LGB full 269K (MSE objective)

Refit TF-IDF char_wb trên full 269K (thay vì 50K subset).
num_leaves=63 (tang tu 31 vi co nhieu data hon).

In [4]:
# Fit TF-IDF char_wb tren full 269K — dung chung cho 5A va 5B
print("Fitting TF-IDF char_wb on 269K docs...")
vec_main = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_main = vec_main.fit_transform(documents)
print(f"Feature matrix: {X_main.shape}")

Fitting TF-IDF char_wb on 269K docs...
Feature matrix: (269112, 100000)


In [ ]:
# 5A. LGB MSE — full 269K
# num_leaves=63 de tang capacity voi 5x data so voi benchmark (50K)
print("[5A] Training LGB (MSE, full 269K)...")
lgb_mse = lgb.LGBMRegressor(
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
lgb_mse.fit(X_main, prices)
print("Training done.")

def lgb_mse_pricer(item):
    x = vec_main.transform([item.summary])
    return max(5, lgb_mse.predict(x)[0])

print("Evaluating [5A]:")
results["5A. LGB MSE (269K)"] = evaluate(lgb_mse_pricer, test)

# train 38 phut

[5A] Training LGB (MSE, full 269K)...
Training done.
Evaluating [5A]:


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  0%|          | 0/200 [00:00<?, ?it/s]

127 7 80 80 14 218 90 43 234 20 21 54 239 99 5 98 2 139 36 46 191 43 51 40 77 16 122 1 581 101 15 108 9 63 7 81 4 58 506 382 151 62 89 5 189 302 56 17 2 84 28 25 154 387 30 66 93 24 104 257 178 6 26 105 29 12 35 40 10 6 15 30 95 110 32 157 146 77 39 51 22 549 54 152 101 84 32 31 89 27 40 25 65 432 4 21 90 147 52 55 135 16 70 123 128 82 18 38 73 9 131 23 327 292 78 24 136 244 22 31 45 37 45 19 24 15 27 135 154 17 60 30 199 18 6 194 102 19 121 34 15 136 94 13 41 88 18 88 23 87 317 117 120 167 61 200 68 103 12 15 83 160 149 70 59 244 24 167 41 188 56 25 248 53 31 85 75 33 77 75 189 36 79 186 22 143 63 496 312 47 165 21 9 6 34 20 56 19 59 19 


In [4]:
# Fit TF-IDF char_wb tren full 269K — dung chung cho 5A va 5B
print("Fitting TF-IDF char_wb on 269K docs...")
vec_main = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_main = vec_main.fit_transform(documents)
print(f"Feature matrix: {X_main.shape}")

Fitting TF-IDF char_wb on 269K docs...
Feature matrix: (269112, 100000)


## Section 5B — LGB full 269K (MAE objective)

objective='regression_l1' toi uu MAE truc tiep (thay vi MSE).
Dung lai X_main da fit o 5A.

In [5]:
# 5B. LGB MAE — full 269K, objective='regression_l1'
print("[5B] Training LGB (MAE objective, full 269K)...")
lgb_mae = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    bagging_freq=1,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
lgb_mae.fit(X_main, prices)
print("Training done.")

def lgb_mae_pricer(item):
    x = vec_main.transform([item.summary])
    return max(5, lgb_mae.predict(x)[0])

print("Evaluating [5B]:")
results["5B. LGB MAE obj (269K)"] = evaluate(lgb_mae_pricer, test)

[5B] Training LGB (MAE objective, full 269K)...
Training done.
Evaluating [5B]:


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  0%|          | 0/200 [00:00<?, ?it/s]

235 41 90 121 5 165 60 48 145 2 37 38 360 74 131 45 6 161 93 6 190 79 35 3 151 70 123 23 637 107 22 109 39 26 13 73 35 45 636 449 207 17 128 29 130 339 32 3 15 22 36 20 147 381 18 51 89 22 100 128 217 8 7 77 55 27 20 47 136 12 20 37 84 185 43 175 162 35 67 20 49 546 96 318 42 85 7 5 27 31 94 33 39 465 5 79 61 100 73 7 163 71 53 73 176 27 28 14 11 74 62 6 377 250 24 15 127 243 41 25 20 11 44 0 32 2 21 117 236 22 180 13 253 0 43 168 6 162 42 20 14 61 114 0 9 74 9 114 1 51 299 85 122 125 72 195 108 180 26 18 102 55 123 32 67 129 0 27 40 393 11 26 171 83 16 19 97 15 3 101 220 21 126 77 47 230 21 588 405 54 140 19 16 73 63 10 34 98 80 10 


## Section 5D — RandomForest + BoW 2000 (Hypothesis Test)

Gia thuyet: RF thua o Section 4 vi TF-IDF 100K features qua cao chieu.
Neu dung BoW 2000 (giong English day3), RF co the tot hon.

Config English day3: CountVectorizer(max_features=2000, stop_words='english')
Config Vietnamese: khong dung stop_words (tieng Viet khac)

In [6]:
print("[5D] Fitting CountVect word (1,2) 50K...")

vec_bow_hyp = CountVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=50_000,
)
X_bow = vec_bow_hyp.fit_transform(documents).astype(np.float32)
print(f"  Count matrix: {X_bow.shape}")

[5D] Fitting CountVect word (1,2) 50K...
  Count matrix: (269112, 50000)


In [7]:
# 5D. RF + CountVect word (1,2) 50K — hypothesis test
subset = 15_000
print(f"[5D] Training RF (100 trees, {subset:,} subset)...")
rf_bow = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_bow.fit(X_bow[:subset], prices[:subset])
print("Training done.")

def rf_bow_pricer(item):
    x = vec_bow_hyp.transform([item.summary]).astype(np.float32)
    return max(5, rf_bow.predict(x)[0])

print("Evaluating [5D]:")
results["5D. RF + CountVect word 50K"] = evaluate(rf_bow_pricer, test)

[5D] Training RF (100 trees, 15,000 subset)...
Training done.
Evaluating [5D]:


  0%|          | 0/200 [00:00<?, ?it/s]

186 81 92 47 68 335 1 23 198 139 1 14 323 86 61 37 7 151 56 9 146 221 75 106 437 4 12 10 568 110 69 260 167 61 103 158 12 20 602 519 146 20 162 9 178 293 74 8 40 49 217 25 213 419 121 82 122 19 122 111 158 4 24 46 130 4 19 105 272 20 84 49 7 179 108 85 0 40 49 52 87 433 52 514 63 229 37 17 176 42 42 93 10 547 16 93 155 358 176 179 75 98 52 14 14 18 65 121 45 9 21 43 452 124 57 68 108 556 60 24 101 18 64 21 123 17 19 81 154 100 187 35 338 2 36 117 118 91 173 62 39 97 41 2 101 139 106 202 51 80 386 68 302 466 71 369 43 230 1 86 24 139 80 330 43 98 35 89 13 325 94 87 119 154 58 188 226 17 434 54 214 16 150 145 41 267 37 576 152 32 225 113 67 53 53 26 81 198 26 59 


## Section 5E — XGBoost + BoW 2000 (Hypothesis Test)

Gia thuyet: XGBoost thua o Section 4 vi TF-IDF 100K features.
BoW 2000 (giong English day3) se cho ket qua tot hon.
Khong can colsample_bytree vi chi 2000 features.

In [8]:
# 5E. XGBoost + CountVect word (1,2) 50K — hypothesis test
# Dung lai vec_bow_hyp va X_bow tu 5D
print("[5E] Training XGBoost (CountVect word 50K, full 269K)...")
np.random.seed(42)
xgb_bow = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)
xgb_bow.fit(X_bow, prices)
print("Training done.")

def xgb_bow_pricer(item):
    x = vec_bow_hyp.transform([item.summary]).astype(np.float32)
    return max(5, xgb_bow.predict(x)[0])

print("Evaluating [5E]:")
results["5E. XGBoost + CountVect word 50K"] = evaluate(xgb_bow_pricer, test)

[5E] Training XGBoost (CountVect word 50K, full 269K)...
Training done.
Evaluating [5E]:


  0%|          | 0/200 [00:00<?, ?it/s]

179 43 168 28 9 255 177 53 287 153 2 47 177 109 2 7 103 134 152 13 93 69 134 21 273 17 103 9 582 73 102 169 111 75 118 110 20 25 618 373 91 25 167 6 118 265 59 13 27 24 154 30 135 422 169 6 126 29 22 226 236 50 55 88 46 11 61 30 112 42 37 61 23 90 92 91 63 27 22 31 16 400 94 366 57 131 41 60 205 46 17 55 121 509 84 27 93 221 119 121 208 30 36 60 86 45 32 19 7 12 50 56 425 292 159 56 58 500 32 20 35 89 25 18 7 31 42 89 286 46 227 57 220 1 83 208 166 19 88 51 82 167 26 8 130 163 27 158 105 40 243 134 191 428 171 228 38 107 20 118 18 122 69 214 86 144 13 163 13 404 31 44 122 68 31 95 207 47 224 85 237 58 106 106 21 226 37 486 412 6 160 72 10 36 32 60 99 255 32 84 


## Section 5F — LGB + CountVectorizer word (1,2) 50K (MSE)

So sanh voi 5A: thay TF-IDF char_wb bang CountVectorizer word-level.
- CountVect word (1,2) 50K: raw count, khong IDF weighting
- TF-IDF char_wb (2,4) 100K: co IDF weighting, char-level features
Ket qua cho thay IDF weighting va char ngrams co giup ich khong.

In [4]:
# Fit CountVectorizer word-level (1,2) 50K — dung chung cho 5F va 5G
print("Fitting CountVectorizer word (1,2) 50K on 269K docs...")
vec_count = CountVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=50_000,
)
X_count = vec_count.fit_transform(documents).astype(np.float32)
print(f"  Count matrix: {X_count.shape}")

Fitting CountVectorizer word (1,2) 50K on 269K docs...
  Count matrix: (269112, 50000)


In [6]:
# Fit CountVectorizer word-level (1,2) 50K — dung chung cho 5F va 5G

# 5F. LGB MSE — CountVectorizer word (1,2) 50K, full 269K
print("[5F] Training LGB (MSE, CountVect word 50K)...")
lgb_count_mse = lgb.LGBMRegressor(
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
)
lgb_count_mse.fit(X_count, prices)
print("Training done.")

def lgb_count_mse_pricer(item):
    x = vec_count.transform([item.summary]).astype(np.float32)
    return max(5, lgb_count_mse.predict(x)[0])

print("Evaluating [5F]:")
results["5F. LGB + CountVect word 50K (MSE)"] = evaluate(lgb_count_mse_pricer, test)

[5F] Training LGB (MSE, CountVect word 50K)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 11.730722 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 139442
[LightGBM] [Info] Number of data points in the train set: 269112, number of used features: 48307
[LightGBM] [Info] Start training from score 330.472996
Training done.
Evaluating [5F]:


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  0%|          | 0/200 [00:00<?, ?it/s]

156 49 128 14 40 296 60 37 323 32 17 7 186 100 16 49 25 108 115 23 56 55 47 69 257 34 177 22 582 63 44 83 36 5 32 40 10 43 552 275 221 66 169 20 192 359 44 12 31 5 60 26 87 337 29 36 127 19 86 357 192 108 24 99 82 12 8 93 37 17 20 35 9 112 51 49 143 34 25 77 79 553 38 304 143 110 17 7 85 24 79 25 61 439 59 42 72 115 108 34 235 2 83 140 29 158 18 81 50 24 28 36 303 329 18 34 48 389 25 33 28 30 18 25 30 1 12 62 139 10 202 10 210 5 113 220 187 125 129 70 28 73 52 13 40 87 3 58 138 81 202 50 175 305 67 122 6 6 0 70 38 48 114 89 7 195 1 311 39 251 92 29 109 130 78 19 219 27 41 14 188 5 8 190 8 66 123 598 505 28 77 51 50 15 64 84 14 73 21 12 


In [7]:
# 5G. LGB MAE — CountVectorizer word (1,2) 50K, full 269K
# Dung lai vec_count va X_count tu 5F
print("[5G] Training LGB (MAE objective, CountVect word 50K)...")
lgb_count_mae = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1000,
    num_leaves=63,
    learning_rate=0.1,
    min_child_samples=30,
    n_jobs=-1,
    random_state=42,
)
lgb_count_mae.fit(X_count, prices)
print("Training done.")

def lgb_count_mae_pricer(item):
    x = vec_count.transform([item.summary]).astype(np.float32)
    return max(5, lgb_count_mae.predict(x)[0])

print("Evaluating [5G]:")
results["5G. LGB + CountVect word 50K (MAE)"] = evaluate(lgb_count_mae_pricer, test)

[5G] Training LGB (MAE objective, CountVect word 50K)...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 13.345023 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 139442
[LightGBM] [Info] Number of data points in the train set: 269112, number of used features: 48307
[LightGBM] [Info] Start training from score 215.000000
Training done.
Evaluating [5G]:


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  0%|          | 0/200 [00:00<?, ?it/s]

242 11 78 55 41 291 146 58 184 29 0 24 261 121 27 74 5 120 48 37 153 50 34 24 290 27 95 3 609 31 11 59 9 5 18 98 1 6 653 349 138 28 184 60 266 358 13 27 33 10 52 32 145 384 44 50 160 21 29 131 148 124 8 55 25 51 31 101 8 7 10 56 15 143 32 15 24 52 52 100 5 552 59 310 44 121 22 1 116 9 30 21 29 519 26 60 45 277 65 64 185 17 91 38 6 39 21 4 27 34 27 11 363 304 57 8 112 517 1 55 54 43 51 24 25 10 15 20 211 34 252 12 313 0 122 175 83 25 139 72 62 37 20 0 186 56 59 159 73 37 287 99 230 379 54 111 40 117 4 10 103 12 112 96 20 63 57 355 41 310 170 23 137 152 65 39 259 10 177 24 149 9 42 85 0 101 24 657 459 66 103 60 58 86 67 23 18 147 9 21 
